# Step 1: Environment Setup
We begin by cloning the **SwinIR** repository, which contains the core architecture for our Stage 1 super-resolution model. We also import essential libraries like `torch` for deep learning, `cv2` for image processing, and `matplotlib` for visualization.

In [ ]:
# CELL 1: RUN THIS FIRST
!git clone https://github.com/JingyunLiang/SwinIR.git
import sys
sys.path.append('/kaggle/working/SwinIR')

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Ensure Kaggle uses the GPU!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

# Step 2: Install Dependencies
We install `rasterio`, a library specifically designed for reading and writing satellite imagery formats like GeoTIFF, which are used in the WorldStrat dataset.

In [ ]:
!pip install rasterio

# Step 3: Data Preprocessing
In this step, we process the raw WorldStrat dataset. We extract the Red, Green, and Blue bands from Sentinel-2 TIFF files, apply a brightness gain to align them visually, and save them as PNG pairs. This prepares the data for training the SwinIR model.

In [ ]:
import os
import rasterio
import numpy as np
import cv2
import shutil
from tqdm import tqdm

# 1. PATH CONFIGURATION
raw_dir = "/kaggle/input/datasets/jucor1/worldstrat/"
hr_base = os.path.join(raw_dir, "hr_dataset", "12bit")
lr_base = os.path.join(raw_dir, "lr_dataset")

clean_lr_dir = "/kaggle/working/cleaned_dataset/lr_dir"
clean_hr_dir = "/kaggle/working/cleaned_dataset/hr_dir"

os.makedirs(clean_lr_dir, exist_ok=True)
os.makedirs(clean_hr_dir, exist_ok=True)

def verify_image(filepath):
    """Checks if the image file is actually readable and not empty."""
    img = cv2.imread(filepath)
    if img is None or img.size == 0:
        return False
    return True

def process_lr_tiff(tiff_path, output_filepath):
    try:
        with rasterio.open(tiff_path) as src:
            # Sentinel-2 Band Order: 4=Red, 3=Green, 2=Blue
            red = src.read(4).astype(np.float32)
            green = src.read(3).astype(np.float32)
            blue = src.read(2).astype(np.float32)
            
        gain = 1.5 
        red = np.clip(red * 255.0 * gain, 0, 255)
        green = np.clip(green * 255.0 * gain, 0, 255)
        blue = np.clip(blue * 255.0 * gain, 0, 255)
        
        bgr_image = np.dstack((blue, green, red)).astype(np.uint8)
        
        # Save and then immediately verify it wrote correctly
        cv2.imwrite(output_filepath, bgr_image)
        return verify_image(output_filepath)
    except Exception:
        return False

# 3. EXECUTION LOOP
locations = [d for d in os.listdir(hr_base) if os.path.isdir(os.path.join(hr_base, d))]

print(f"🔄 Processing and Validating {len(locations)} locations...")

valid_count = 0
for loc in tqdm(locations):
    hr_image_path = os.path.join(hr_base, loc, f"{loc}_rgb.png")
    lr_tiff_path = os.path.join(lr_base, loc, "L2A", f"{loc}-1-L2A_data.tiff")
    
    if os.path.exists(hr_image_path) and os.path.exists(lr_tiff_path):
        out_hr = os.path.join(clean_hr_dir, f"{loc}.png")
        out_lr = os.path.join(clean_lr_dir, f"{loc}.png")
        
        # Check source HR before copying
        if not verify_image(hr_image_path):
            continue # Skip this location entirely if HR is broken
            
        # Copy HR and Process LR
        shutil.copy(hr_image_path, out_hr)
        success_lr = process_lr_tiff(lr_tiff_path, out_lr)
        
        # FINAL SANITY CHECK: If LR failed, delete the HR copy to prevent orphans
        if not success_lr:
            if os.path.exists(out_hr): os.remove(out_hr)
            if os.path.exists(out_lr): os.remove(out_lr)
        else:
            valid_count += 1

print(f"✅ Finished! Generated {valid_count} healthy image pairs.")

# Step 4: Data Validation & Visualization
Before training, it's crucial to verify the image pairs. Here, we randomly sample and visualize the Low-Resolution (LR) and High-Resolution (HR) ground truth pairs to ensure they are correctly aligned and processed.

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt

lr_dir = "/kaggle/working/cleaned_dataset/lr_dir"
hr_dir = "/kaggle/working/cleaned_dataset/hr_dir"

# Get common filenames (safety check)
lr_files = set(os.listdir(lr_dir))
hr_files = set(os.listdir(hr_dir))

common_files = list(lr_files.intersection(hr_files))

print("Total matched pairs:", len(common_files))

# Pick 10 random samples
samples = random.sample(common_files, min(10, len(common_files)))

for file in samples:
    lr_path = os.path.join(lr_dir, file)
    hr_path = os.path.join(hr_dir, file)

    # Load images
    lr_img = cv2.imread(lr_path)
    hr_img = cv2.imread(hr_path)

    # Convert BGR → RGB for correct display
    lr_img = cv2.cvtColor(lr_img, cv2.COLOR_BGR2RGB)
    hr_img = cv2.cvtColor(hr_img, cv2.COLOR_BGR2RGB)

    # Resize LR to match HR size (for visual comparison)
    lr_resized = cv2.resize(lr_img, (hr_img.shape[1], hr_img.shape[0]))

    # Plot side by side
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(lr_resized)
    plt.title(f"LR (resized)\n{file}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(hr_img)
    plt.title("HR (Ground Truth)")
    plt.axis("off")

    plt.show()

# Step 5: PyTorch Dataset & DataLoader
We define a custom `WorldStratDataset` class to handle the data pipeline. This class performs on-the-fly resizing (LR to 128x128, HR to 512x512) and normalization, ensuring the model receives batches of tensors ready for training.

In [ ]:
import random

class WorldStratDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, lr_size=128, hr_size=512):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.lr_size = lr_size 
        self.hr_size = hr_size
        
        valid_filenames = []
        for f in os.listdir(lr_dir):
            if f.endswith('.png'):
                if os.path.exists(os.path.join(hr_dir, f)):
                    valid_filenames.append(f)
                    
        self.image_filenames = valid_filenames
        print(f"📦 Dataset initialized with {len(self.image_filenames)} potential pairs.")
        
    def __len__(self):
        return len(self.image_filenames)
        
    def __getitem__(self, idx):
        try:
            img_name = self.image_filenames[idx]
            lr_path = os.path.join(self.lr_dir, img_name)
            hr_path = os.path.join(self.hr_dir, img_name)
            
            lr_img = cv2.imread(lr_path)
            hr_img = cv2.imread(hr_path)
            
            # If OpenCV fails to read (corrupted file), try a random different index
            if lr_img is None or hr_img is None:
                print(f"⚠️ Skipping corrupted file: {img_name}")
                return self.__getitem__(random.randint(0, len(self.image_filenames) - 1))
            
            # Standard processing
            lr_img = cv2.resize(lr_img, (self.lr_size, self.lr_size), interpolation=cv2.INTER_AREA)
            hr_img = cv2.resize(hr_img, (self.hr_size, self.hr_size), interpolation=cv2.INTER_CUBIC)
            
            lr_img = cv2.cvtColor(lr_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            hr_img = cv2.cvtColor(hr_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            
            return torch.from_numpy(lr_img).permute(2, 0, 1), torch.from_numpy(hr_img).permute(2, 0, 1)
            
        except Exception as e:
            # If any other error occurs (like libpng read error), grab a different image
            return self.__getitem__(random.randint(0, len(self.image_filenames) - 1))

# Setup dataloader (Batch size 2 to prevent Out Of Memory)
dataset = WorldStratDataset("/kaggle/working/cleaned_dataset/lr_dir", "/kaggle/working/cleaned_dataset/hr_dir", lr_size=128, hr_size=512)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2)

# Step 6: Model Dependencies
We install `timm` (PyTorch Image Models), which is often required by transformer-based vision architectures like SwinIR.

In [ ]:
!pip install timm

# Step 7: SwinIR Model Configuration
We initialize the **SwinIR** model. Following the paper's Stage 1 architecture, we configure it for 4x upscaling using Swin Transformer blocks. We also set up the Adam optimizer and a learning rate scheduler.

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim

if '/kaggle/working/SwinIR' not in sys.path:
    sys.path.append('/kaggle/working/SwinIR')

from models.network_swinir import SwinIR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

model = SwinIR(
    upscale=4,
    in_chans=3,
    img_size=128,
    window_size=8,
    img_range=1., 
    depths=[6, 6, 6, 6],
    embed_dim=60,
    num_heads=[6, 6, 6, 6],
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv'
).to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=2e-4)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

# Step 8: Training Loop
This is the main training execution. We use **Mixed Precision Training (AMP)** to optimize memory usage and speed on Kaggle's GPUs. The loop includes loss calculation (L1 Loss), backpropagation, and model checkpointing to save the best weights.

In [ ]:
import os
import gc
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm

# -----------------------------
# 1) SETUP
# -----------------------------
torch.cuda.empty_cache()
gc.collect()

torch.backends.cudnn.benchmark = True  # good when input sizes stay the same

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

batch_size = 2
num_workers = 2

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(num_workers > 0)
)

# -----------------------------
# 2) SAVE PATHS
# -----------------------------
save_dir = "/kaggle/working/model_saves"
os.makedirs(save_dir, exist_ok=True)

latest_model_path = os.path.join(save_dir, "model_latest.pth")
best_model_path = os.path.join(save_dir, "model_best.pth")

# -----------------------------
# 3) TRAINING
# -----------------------------
num_epochs = 50
best_loss = float("inf")

print(f"Starting training for {num_epochs} epochs...")

use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{num_epochs}")

    for lr_imgs, hr_imgs in pbar:
        lr_imgs = lr_imgs.to(device, non_blocking=True)
        hr_imgs = hr_imgs.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(lr_imgs)
            loss = criterion(outputs, hr_imgs)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch} done | Avg Loss: {avg_loss:.6f}")

    # Save latest model weights only
    torch.save(model.state_dict(), latest_model_path)

    # Save best model weights only
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model updated at epoch {epoch} | Loss: {best_loss:.6f}")

print("Training finished.")
print(f"Latest model saved at: {latest_model_path}")
print(f"Best model saved at:   {best_model_path}")

# Step 9: INFERENCE + EVALUATION

In [ ]:
# Loads best model, runs on test images, measures quality

import os
import sys
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

# ─────────────────────────────────────────
# STEP 1: LOAD YOUR BEST TRAINED MODEL
# ─────────────────────────────────────────
if '/kaggle/working/SwinIR' not in sys.path:
    sys.path.append('/kaggle/working/SwinIR')
from models.network_swinir import SwinIR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

# Build same model architecture as training
model = SwinIR(
    upscale=4, in_chans=3, img_size=128, window_size=8, img_range=1.,
    depths=[6, 6, 6, 6], embed_dim=60, num_heads=[6, 6, 6, 6],
    mlp_ratio=2, upsampler='pixelshuffle', resi_connection='1conv'
).to(device)

# Load the best saved weights
best_model_path = "/kaggle/working/model_saves/model_best.pth"
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()  # ← CRITICAL: switches off dropout/batchnorm training behavior
print("✅ Best model loaded successfully!")

# ─────────────────────────────────────────
# STEP 2: BUILD A SMALL TEST SET
# Held-out images the model has NEVER seen
# ─────────────────────────────────────────
lr_dir = "/kaggle/working/cleaned_dataset/lr_dir"
hr_dir = "/kaggle/working/cleaned_dataset/hr_dir"

all_files = sorted([
    f for f in os.listdir(lr_dir)
    if f.endswith('.png') and os.path.exists(os.path.join(hr_dir, f))
])

# Take last 5% of files as test set
# Important: these should be files NOT used in training
n_test = max(10, int(len(all_files) * 0.05))
test_files = all_files[-n_test:]  # last N files
print(f"📊 Evaluating on {len(test_files)} test images...")

# ─────────────────────────────────────────
# STEP 3: RUN INFERENCE + COLLECT METRICS
# ─────────────────────────────────────────
psnr_scores = []
ssim_scores = []

results = []  # store images for visualization later

for filename in tqdm(test_files):
    lr_path = os.path.join(lr_dir, filename)
    hr_path = os.path.join(hr_dir, filename)

    # --- Load Images ---
    lr_img = cv2.imread(lr_path)
    hr_img = cv2.imread(hr_path)

    if lr_img is None or hr_img is None:
        continue

    # --- Preprocess LR (same as training) ---
    lr_resized = cv2.resize(lr_img, (128, 128), interpolation=cv2.INTER_AREA)
    lr_rgb = cv2.cvtColor(lr_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    # Shape: (H, W, C) → (1, C, H, W)  [batch of 1]
    lr_tensor = torch.from_numpy(lr_rgb).permute(2, 0, 1).unsqueeze(0).to(device)

    # --- Run Model (no gradient needed = faster + less memory) ---
    with torch.no_grad():
        sr_tensor = model(lr_tensor)

    # --- Convert Output Back to Image ---
    # Shape: (1, C, H, W) → (H, W, C)
    sr_img = sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    sr_img = np.clip(sr_img, 0, 1)  # ensure valid range

    # --- Prepare HR for comparison ---
    hr_resized = cv2.resize(hr_img, (512, 512), interpolation=cv2.INTER_CUBIC)
    hr_rgb = cv2.cvtColor(hr_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    # --- Calculate PSNR ---
    # "How similar are the pixel values?"
    # Higher = better (30+ is good, 40+ is excellent)
    psnr_val = psnr_metric(hr_rgb, sr_img, data_range=1.0)
    psnr_scores.append(psnr_val)

    # --- Calculate SSIM ---
    # "How similar are the structures/textures?"
    # Range: 0 to 1 (higher = better)
    ssim_val = ssim_metric(hr_rgb, sr_img, data_range=1.0, channel_axis=2)
    ssim_scores.append(ssim_val)

    # Store for visualization (only first 5)
    if len(results) < 5:
        results.append({
            'filename': filename,
            'lr': lr_rgb,
            'sr': sr_img,
            'hr': hr_rgb,
            'psnr': psnr_val,
            'ssim': ssim_val
        })

# ─────────────────────────────────────────
# STEP 4: PRINT SUMMARY METRICS
# ─────────────────────────────────────────
avg_psnr = np.mean(psnr_scores)
avg_ssim = np.mean(ssim_scores)

print("\n" + "="*50)
print("📊 EVALUATION RESULTS")
print("="*50)
print(f"Average PSNR:  {avg_psnr:.4f} dB")
print(f"Average SSIM:  {avg_ssim:.4f}")
print(f"Best PSNR:     {max(psnr_scores):.4f} dB")
print(f"Worst PSNR:    {min(psnr_scores):.4f} dB")
print("="*50)

# Interpret results for you
if avg_psnr >= 30:
    print("✅ PSNR is good! Model is performing well.")
elif avg_psnr >= 25:
    print("⚠️  PSNR is okay but could improve. Consider more epochs.")
else:
    print("❌ PSNR is low. Model may need more training or tuning.")

if avg_ssim >= 0.8:
    print("✅ SSIM is excellent! Structures are well preserved.")
elif avg_ssim >= 0.6:
    print("⚠️  SSIM is moderate. Some structural details may be lost.")
else:
    print("❌ SSIM is low. Model struggling with structural details.")

# ─────────────────────────────────────────
# STEP 5: VISUALIZE RESULTS
# LR (blurry) | SR (our output) | HR (ground truth)
# ─────────────────────────────────────────
print("\n📸 Visualizing sample results...")

for i, result in enumerate(results):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Show LR resized to same display size
    lr_display = cv2.resize(result['lr'], (512, 512))

    axes[0].imshow(lr_display)
    axes[0].set_title("LR Input\n(Blurry Sentinel)", fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(result['sr'])
    axes[1].set_title(
        f"SR Output (Ours)\nPSNR: {result['psnr']:.2f} | SSIM: {result['ssim']:.3f}",
        fontsize=12
    )
    axes[1].axis('off')

    axes[2].imshow(result['hr'])
    axes[2].set_title("HR Ground Truth\n(Sharp SPOT)", fontsize=12)
    axes[2].axis('off')

    plt.suptitle(f"Sample {i+1}: {result['filename']}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/result_sample_{i+1}.png", dpi=150, bbox_inches='tight')
    plt.show()

print("✅ All results saved to /kaggle/working/")

In [ ]:
# ============================================================
# CELL 8 — HISTOGRAM MATCHING + RE-EVALUATION
# Apply HM to fix color shift, then re-evaluate
# ============================================================

import os
import sys
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
from scipy.interpolate import interp1d

# ─────────────────────────────────────────
# STEP 1: HISTOGRAM MATCHING FUNCTION
# Adapted from skimage histogram matching
# ─────────────────────────────────────────

def histogram_matching(source, reference):
    """
    Adjust source image so its histogram matches reference image.
    
    Args:
        source: numpy array, shape (H, W, C), values 0-1
        reference: numpy array, shape (H, W, C), values 0-1
    
    Returns:
        matched: numpy array, same shape as source, values 0-1
    """
    
    def _match_histograms_1d(source_1d, reference_1d):
        """Match histogram of a single channel"""
        # Calculate cumulative distributions (CDFs)
        src_hist, src_bins = np.histogram(source_1d, bins=256, range=(0, 1))
        ref_hist, ref_bins = np.histogram(reference_1d, bins=256, range=(0, 1))
        
        # Convert to CDF (cumulative distribution)
        src_cdf = np.cumsum(src_hist).astype(float)
        src_cdf /= src_cdf[-1]  # Normalize to 0-1
        
        ref_cdf = np.cumsum(ref_hist).astype(float)
        ref_cdf /= ref_cdf[-1]  # Normalize to 0-1
        
        # Create mapping function
        # "For each percentile in source, what value in reference is at that percentile?"
        ref_values = np.linspace(0, 1, len(ref_cdf))
        src_values = np.linspace(0, 1, len(src_cdf))
        
        # Interpolation: map src CDF values to ref values
        f = interp1d(src_cdf, src_values, bounds_error=False, 
                     fill_value="extrapolate")
        g = interp1d(ref_cdf, ref_values, bounds_error=False, 
                     fill_value="extrapolate")
        
        # Get the matched output
        matched = g(f(source_1d))
        return np.clip(matched, 0, 1)
    
    # Apply to each channel independently
    matched = np.zeros_like(source)
    for channel in range(source.shape[2]):
        matched[:, :, channel] = _match_histograms_1d(
            source[:, :, channel].flatten(),
            reference[:, :, channel].flatten()
        ).reshape(source[:, :, channel].shape)
    
    return matched

# ─────────────────────────────────────────
# STEP 2: LOAD MODEL (same as before)
# ─────────────────────────────────────────
if '/kaggle/working/SwinIR' not in sys.path:
    sys.path.append('/kaggle/working/SwinIR')
from models.network_swinir import SwinIR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

model = SwinIR(
    upscale=4, in_chans=3, img_size=128, window_size=8, img_range=1.,
    depths=[6, 6, 6, 6], embed_dim=60, num_heads=[6, 6, 6, 6],
    mlp_ratio=2, upsampler='pixelshuffle', resi_connection='1conv'
).to(device)

best_model_path = "/kaggle/input/models/syedabdullah1247/best-model/pytorch/default/1/model_best.pth"
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print("✅ Model loaded!")

# ─────────────────────────────────────────
# STEP 3: RUN INFERENCE WITH HM
# ─────────────────────────────────────────
lr_dir = "/kaggle/working/cleaned_dataset/lr_dir"
hr_dir = "/kaggle/working/cleaned_dataset/hr_dir"

all_files = sorted([
    f for f in os.listdir(lr_dir)
    if f.endswith('.png') and os.path.exists(os.path.join(hr_dir, f))
])

n_test = max(10, int(len(all_files) * 0.05))
test_files = all_files[-n_test:]

print(f"🎨 Processing {len(test_files)} images with Histogram Matching...")

# Store results for comparison
psnr_before_hm = []
psnr_after_hm = []
ssim_before_hm = []
ssim_after_hm = []

comparison_results = []

for filename in tqdm(test_files):
    lr_path = os.path.join(lr_dir, filename)
    hr_path = os.path.join(hr_dir, filename)

    lr_img = cv2.imread(lr_path)
    hr_img = cv2.imread(hr_path)

    if lr_img is None or hr_img is None:
        continue

    # --- Inference (same as before) ---
    lr_resized = cv2.resize(lr_img, (128, 128), interpolation=cv2.INTER_AREA)
    lr_rgb = cv2.cvtColor(lr_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    lr_tensor = torch.from_numpy(lr_rgb).permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        sr_tensor = model(lr_tensor)

    sr_img = sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    sr_img = np.clip(sr_img, 0, 1)

    # --- Prepare HR ---
    hr_resized = cv2.resize(hr_img, (512, 512), interpolation=cv2.INTER_CUBIC)
    hr_rgb = cv2.cvtColor(hr_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    # --- BEFORE HM: Calculate metrics on raw SR ---
    psnr_before = psnr_metric(hr_rgb, sr_img, data_range=1.0)
    ssim_before = ssim_metric(hr_rgb, sr_img, data_range=1.0, channel_axis=2)

    psnr_before_hm.append(psnr_before)
    ssim_before_hm.append(ssim_before)

    # --- APPLY HISTOGRAM MATCHING ---
    # This is the KEY step from the paper!
    sr_hm = histogram_matching(sr_img, hr_rgb)
    sr_hm = np.clip(sr_hm, 0, 1)

    # --- AFTER HM: Calculate metrics on color-corrected SR ---
    psnr_after = psnr_metric(hr_rgb, sr_hm, data_range=1.0)
    ssim_after = ssim_metric(hr_rgb, sr_hm, data_range=1.0, channel_axis=2)

    psnr_after_hm.append(psnr_after)
    ssim_after_hm.append(ssim_after)

    # Store for visualization (first 5 only)
    if len(comparison_results) < 5:
        comparison_results.append({
            'filename': filename,
            'lr': lr_rgb,
            'sr_before': sr_img,
            'sr_after': sr_hm,
            'hr': hr_rgb,
            'psnr_before': psnr_before,
            'psnr_after': psnr_after,
            'ssim_before': ssim_before,
            'ssim_after': ssim_after
        })

# ─────────────────────────────────────────
# STEP 4: PRINT COMPARISON RESULTS
# ─────────────────────────────────────────
avg_psnr_before = np.mean(psnr_before_hm)
avg_psnr_after = np.mean(psnr_after_hm)
avg_ssim_before = np.mean(ssim_before_hm)
avg_ssim_after = np.mean(ssim_after_hm)

psnr_improvement = avg_psnr_after - avg_psnr_before
ssim_improvement = avg_ssim_after - avg_ssim_before

print("\n" + "="*60)
print("📊 HISTOGRAM MATCHING IMPACT")
print("="*60)
print(f"\n{'Metric':<15} {'Before HM':<15} {'After HM':<15} {'Improvement':<15}")
print("-"*60)
print(f"{'PSNR (dB)':<15} {avg_psnr_before:<15.4f} {avg_psnr_after:<15.4f} {psnr_improvement:+.4f}")
print(f"{'SSIM':<15} {avg_ssim_before:<15.4f} {avg_ssim_after:<15.4f} {ssim_improvement:+.4f}")
print("="*60)

# Interpretation
if psnr_improvement > 2:
    print(f"✅ EXCELLENT! PSNR improved by {psnr_improvement:.2f} dB")
elif psnr_improvement > 0.5:
    print(f"✅ GOOD! PSNR improved by {psnr_improvement:.2f} dB")
elif psnr_improvement > 0:
    print(f"⚠️  Modest improvement of {psnr_improvement:.2f} dB")
else:
    print(f"❌ Histogram Matching made things worse by {abs(psnr_improvement):.2f} dB")

if ssim_improvement > 0.05:
    print(f"✅ EXCELLENT! SSIM improved by {ssim_improvement:.4f}")
elif ssim_improvement > 0.01:
    print(f"✅ GOOD! SSIM improved by {ssim_improvement:.4f}")
elif ssim_improvement > 0:
    print(f"⚠️  Modest SSIM improvement of {ssim_improvement:.4f}")
else:
    print(f"❌ SSIM degraded by {abs(ssim_improvement):.4f}")

print("\n💡 What this means:")
print("   Histogram Matching corrects color/brightness mismatch between")
print("   Sentinel (LR) and SPOT (HR). This is crucial preprocessing for SR3.")

# ─────────────────────────────────────────
# STEP 5: VISUALIZATION - BEFORE vs AFTER HM
# ─────────────────────────────────────────
print("\n📸 Visualizing Before/After Histogram Matching...")

for idx, result in enumerate(comparison_results):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # ===== TOP ROW: Before HM =====
    lr_display = cv2.resize(result['lr'], (512, 512))

    axes[0, 0].imshow(lr_display)
    axes[0, 0].set_title("LR Input\n(Blurry Sentinel)", fontsize=11, fontweight='bold')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(result['sr_before'])
    axes[0, 1].set_title(
        f"SR (Before HM)\nPSNR: {result['psnr_before']:.2f}\nSSIM: {result['ssim_before']:.3f}",
        fontsize=11, fontweight='bold', color='red'
    )
    axes[0, 1].axis('off')

    axes[0, 2].imshow(result['hr'])
    axes[0, 2].set_title("HR Ground Truth\n(Sharp SPOT)", fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')

    # ===== BOTTOM ROW: After HM =====
    axes[1, 0].text(0.5, 0.5, "↓ APPLY HISTOGRAM MATCHING ↓", 
                   ha='center', va='center', fontsize=14, fontweight='bold',
                   transform=axes[1, 0].transAxes,
                   bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    axes[1, 0].axis('off')

    axes[1, 1].imshow(result['sr_after'])
    axes[1, 1].set_title(
        f"SR (After HM) ✨\nPSNR: {result['psnr_after']:.2f}\nSSIM: {result['ssim_after']:.3f}",
        fontsize=11, fontweight='bold', color='green'
    )
    axes[1, 1].axis('off')

    # Show difference in improvements
    psnr_diff = result['psnr_after'] - result['psnr_before']
    ssim_diff = result['ssim_after'] - result['ssim_before']
    
    diff_text = f"Improvements:\n"
    diff_text += f"PSNR: {psnr_diff:+.2f} dB\n"
    diff_text += f"SSIM: {ssim_diff:+.4f}"
    
    axes[1, 2].text(0.5, 0.5, diff_text,
                   ha='center', va='center', fontsize=12, fontweight='bold',
                   transform=axes[1, 2].transAxes,
                   bbox=dict(boxstyle='round', facecolor='lightgreen' if psnr_diff > 0 else 'lightcoral', alpha=0.5))
    axes[1, 2].axis('off')

    plt.suptitle(f"Sample {idx+1}: {result['filename']}\nHistogram Matching Effect", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/hm_comparison_{idx+1}.png", dpi=150, bbox_inches='tight')
    plt.show()

# ─────────────────────────────────────────
# STEP 6: SAVE COLOR-CORRECTED OUTPUTS
# For use in next step (SR3 refinement)
# ─────────────────────────────────────────
hm_output_dir = "/kaggle/working/hm_corrected_images"
os.makedirs(hm_output_dir, exist_ok=True)

print("\n💾 Saving histogram-matched outputs for SR3 training...")
for filename in tqdm(test_files[:10]):  # Save first 10 as examples
    lr_path = os.path.join(lr_dir, filename)
    hr_path = os.path.join(hr_dir, filename)

    lr_img = cv2.imread(lr_path)
    hr_img = cv2.imread(hr_path)

    if lr_img is None or hr_img is None:
        continue

    lr_resized = cv2.resize(lr_img, (128, 128), interpolation=cv2.INTER_AREA)
    lr_rgb = cv2.cvtColor(lr_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    lr_tensor = torch.from_numpy(lr_rgb).permute(2, 0, 1).unsqueeze(0).to(device)

    with torch.no_grad():
        sr_tensor = model(lr_tensor)

    sr_img = sr_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
    sr_img = np.clip(sr_img, 0, 1)

    hr_resized = cv2.resize(hr_img, (512, 512), interpolation=cv2.INTER_CUBIC)
    hr_rgb = cv2.cvtColor(hr_resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    sr_hm = histogram_matching(sr_img, hr_rgb)
    sr_hm = np.clip(sr_hm, 0, 1)

    # Convert back to BGR for OpenCV and save
    sr_hm_bgr = cv2.cvtColor((sr_hm * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
    output_path = os.path.join(hm_output_dir, f"hm_{filename}")
    cv2.imwrite(output_path, sr_hm_bgr)

print(f"✅ Saved {len(test_files[:10])} HM-corrected images to {hm_output_dir}")

print("\n" + "="*60)
print("🎉 HISTOGRAM MATCHING COMPLETE!")
print("="*60)
print("\n📝 Summary:")
print(f"  • PSNR improved from {avg_psnr_before:.2f} to {avg_psnr_after:.2f} dB")
print(f"  • SSIM improved from {avg_ssim_before:.4f} to {avg_ssim_after:.4f}")
print(f"  • Color distribution now matches HR (Ground Truth)")
print("\n👉 Next step: Train SR3 model for fine detail refinement")
print("="*60)